# Data Integration and Quality Assessment — Owner B

This notebook implements the reproducible cleaning work for the TfNSW EV charger
dataset. It reads the acquisition output from `data/raw/`, preserves raw values,
and writes auditable cleaned intermediates to `data/interim/`.

The current workflow runs through notebooks: acquisition → this notebook →
augmentation → transformation and storage. The last notebook assigns SA4 regions
while loading the augmented DC records into DuckDB.


## Running this notebook

Run `data_acquisition.ipynb` first to create `data/raw/ev_charging_locations.csv`.
Jupyter may start in the repository root or `notebooks/`; paths are resolved upward.
Restart the kernel and run all cells in order. No network access is needed here.

This notebook does not alter `data/raw/`. Its three `data/interim/` outputs are
regenerated rather than committed.


In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def _repo_root() -> Path:
    """Locate the repository root, wherever Jupyter was launched from."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        f"Could not find the repository root (no pyproject.toml above {here})."
    )


PROJECT_ROOT = _repo_root()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "ev_charging_locations.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = INTERIM_DIR / "ev_chargers_cleaned.csv"
COMPONENTS_CSV = INTERIM_DIR / "charger_power_components.csv"
QUALITY_JSON = INTERIM_DIR / "cleaning_quality_summary.json"

RAW_SHA256 = hashlib.sha256(RAW_CSV.read_bytes()).hexdigest()

raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
assert len(raw) == 1958, "Expected the December 2025 source release with 1,958 rows."
raw.head()


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,,,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.81100405,150.8495966,Blacktown City Council,2766,Existing Fast Chargers
2,,,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.5118739,151.669395,Central Darling Shire Council,2350,TfNSW Regional
3,,,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.77410115,151.167035,Ku-ring-gai Council,2070,Existing Destination Chargers
4,,,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.64181886,153.6136326,Byron Shire Council,2481,Existing Destination Chargers


## 1. Raw-data quality profile

Record source blanks, naming variation, and candidate duplicate groups before
cleaning. Blank means empty or whitespace-only; the literal rating `AC` is counted
separately as semantically missing power. Counts and rates are saved in section 5.

The existing `(Station_name, Latitude, Longitude)` key finds candidates, not proven
duplicates: names are often absent, and different networks can share coordinates.
Record IDs identify source records, not unique physical stations or individual plugs.


In [2]:
missing_before = raw.replace(r"^\s*$", pd.NA, regex=True).isna().sum()
operator_counts_before = raw["Operator"].value_counts()
rating_counts_before = raw["Charger_rating"].value_counts()
duplicate_key_columns = ["Station_name", "Latitude", "Longitude"]
duplicate_groups_before = (
    raw.groupby(duplicate_key_columns, dropna=False)
    .size()
    .reset_index(name="group_size")
    .query("group_size > 1")
    .sort_values(duplicate_key_columns)
)

display(missing_before.rename("missing_values").to_frame())
display(raw[["Operator", "Charger_Type", "Charger_rating"]].nunique().rename("unique_values").to_frame())
display(raw["Operator"].str.len().value_counts().sort_index().rename("operator_length_rows").to_frame())
display(duplicate_groups_before)


,missing_values
OBJECTID,1837
Station_name,1438
Station_address,0
Operator,0
Number_of_plugs,0
Charger_Type,0
Charger_rating,0
Latitude,0
Longitude,0
LGANAME,121


,unique_values
Operator,50
Charger_Type,3
Charger_rating,46


,operator_length_rows
Operator,
2,32
3,112
4,267
5,314
6,58
7,151
8,303
9,267
10,26


,Station_name,Latitude,Longitude,group_size
666,,-33.8556219,151.0767955,2
1239,,-34.4079569,150.877135,2
1250,,-34.4366153,150.8632982,2
1445,Albury City Council,-36.0806351,146.9216087,2
1631,Hay Shire Council,-34.5089561,144.8428431,2
1674,Kempsey Shire Council,-30.88653,153.0379572,2
1742,Narara Ecovillage,-33.3931163,151.3286306,2
1785,Quest Griffith,-34.2866312,146.0445254,2


## 2. Operator canonicalization, stable record IDs and data types

`Operator_raw` retains source text. The existing mapping merges documented prefix,
whitespace/case and `Charge Hub` variants. Unrecoverable truncations stay as reported;
`University of` becomes `Unknown/missing` because it is venue text, not a known network.
Neither station names nor operator names are guessed from addresses.

IDs continue to hash the same original fields before any numeric conversion. They
are stable for this source snapshot; changing a source field creates a different ID.
The original columns remain text for audit and compatibility. Typed companion columns
`latitude_numeric`, `longitude_numeric` (`Float64`) and `number_of_plugs_numeric`
(`Int64`) support analysis without changing the original coordinate/plug strings.
CSV does not retain pandas dtypes; consumers must explicitly parse numeric columns.

| Fields | Missing-value and validation policy |
|---|---|
| `OBJECTID` | Retain blanks; populated IDs must be digits. Use the stable record hash as the key. |
| `Station_name` | Retain blanks and original names; neither impute a venue nor drop the row. |
| `Station_address` | Required nonblank text; retain original formatting. |
| `Operator` | Keep raw text, apply the existing map; count `Unknown/missing` as semantically missing. |
| `Latitude`, `Longitude` | Require numeric values within −90…90 and −180…180. Range checks do not establish address accuracy. EPSG:4326 remains the downstream assumption. |
| `Number_of_plugs` | Require a positive whole number; do not sum overlapping site records. |
| `Charger_Type` | Require `AC`, `DC` or `Upcoming`; only exact `DC` enters augmentation. |
| `Charger_rating` | Preserve raw text. Normalize known kW; leave `AC` power unknown and retain mixed configurations in the child table. |
| `LGANAME`, `PCODE`, `Source` | Retain blanks without imputation. Preserve `PCODE` verbatim; `postcode_normalized` removes an explicit `NSW` prefix and must otherwise contain four digits or be blank. Keep postcodes as text. |

Invalid required numeric/categorical values stop output generation with counts,
rather than silently coercing them into missing values. Missing optional fields do
not cause rows to be removed. No address, operator, power or geographic field is
filled from another row during duplicate resolution.


In [3]:
RAW_TO_CANONICAL = {
    "360 EV Charge": "360 EV Charge", "AXCharge": "AXCharge", "Alchemy Charge": "Alchemy Charge",
    "Ampol": "Ampol", "BMW": "BMW", "BP": "BP Australia", "BP Australia ": "BP Australia",
    "CasaCharge": "CasaCharge", "Charge Hub": "ChargeHub", "Charge OS": "Charge OS",
    "ChargeHub": "ChargeHub", "ChargePoint": "ChargePoint", "ChargePost": "ChargePost",
    "Chargefox": "Chargefox", "Chargestar": "Chargestar", "Counties Energy": "Counties Energy",
    "EVE Australia": "EVE Australia", "EV Meter": "EV Meter", "EVNet": "EVNet", "EVSE": "EVSE",
    "EVUp": "EVUp", "EVX": "EVX", "Elanga": "Elanga", "Energy Austra": "Energy Austra",
    "Engie": "Engie", "Everty": "Everty", "Evie": "Evie Networks", "Evie Networks": "Evie Networks",
    "Exploren": "Exploren", "Fast Cities A": "Fast Cities A", "Gentari": "Gentari", "JOLT": "JOLT",
    "NRMA": "NRMA Electric", "NRMA Electric": "NRMA Electric", "Non-Networked": "Non-networked",
    "Non-networked": "Non-networked", "Noodoe": "Noodoe", "PLUS ES": "PLUS ES",
    "PLUS ES Manag": "PLUS ES Manag", "Porsche Destination Charging": "Porsche Destination Charging",
    "Porsche Smart Mobility": "Porsche Smart Mobility", "Saascharge": "Saascharge",
    "Smart Charge": "Smart Charge", "Tesla": "Tesla Motors", "Tesla Motors ": "Tesla Motors",
    "University of": "Unknown/missing", "Viva Energy A": "Viva Energy Australia",
    "Viva Energy Australia": "Viva Energy Australia", "Wevolt": "Wevolt", "Zeus Renewables": "Zeus Renewables",
}

cleaned = raw.copy()
assert set(cleaned["Operator"]) == set(RAW_TO_CANONICAL), "Review the mapping if the acquisition source changes."
cleaned.insert(cleaned.columns.get_loc("Operator") + 1, "Operator_raw", cleaned["Operator"])
cleaned["Operator"] = cleaned["Operator_raw"].map(RAW_TO_CANONICAL)
assert cleaned["Operator"].notna().all()

ID_FIELDS = [
    "OBJECTID", "Station_name", "Station_address", "Operator_raw", "Number_of_plugs",
    "Charger_Type", "Charger_rating", "Latitude", "Longitude", "LGANAME", "PCODE", "Source",
]

def make_charger_record_id(row: pd.Series) -> str:
    payload = ["charger_record_id_v1", [(field, row[field]) for field in ID_FIELDS]]
    return "chr_" + hashlib.sha256(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

cleaned.insert(0, "charger_record_id", cleaned.apply(make_charger_record_id, axis=1))
assert cleaned["charger_record_id"].is_unique
print(f"Operators: {raw['Operator'].nunique()} raw -> {cleaned['Operator'].nunique()} canonical")


# Validate numeric values in companion columns; ID inputs stay unchanged.
NUMERIC_COLUMNS = {
    "Latitude": ("latitude_numeric", "Float64", -90, 90),
    "Longitude": ("longitude_numeric", "Float64", -180, 180),
    "Number_of_plugs": ("number_of_plugs_numeric", "Int64", 1, None),
}
numeric_validation = {}
for source, (target, dtype, lower, upper) in NUMERIC_COLUMNS.items():
    text_values = raw[source].str.strip()
    values = pd.to_numeric(text_values, errors="coerce")
    missing = text_values.eq("")
    non_numeric = ~missing & values.isna()
    valid = values.ge(lower)
    if upper is not None:
        valid &= values.le(upper)
    else:
        valid &= values.mod(1).eq(0)  # Reject fractions and infinities.
    invalid_domain = values.notna() & ~valid
    numeric_validation[source] = {
        "missing": int(missing.sum()),
        "non_numeric": int(non_numeric.sum()),
        "out_of_range_or_non_integer": int(invalid_domain.sum()),
        "invalid_total": int((missing | non_numeric | invalid_domain).sum()),
        "validated_dtype": dtype,
        "output_column": target,
    }
    assert numeric_validation[source]["invalid_total"] == 0, numeric_validation[source]
    cleaned[target] = values.astype(dtype)

# Ten source postcodes contain an explicit NSW prefix; remove only that prefix
# in a companion field, without inferring missing postcodes from addresses.
postcode_text = raw["PCODE"].str.strip()
cleaned["postcode_normalized"] = postcode_text.str.replace(r"^NSW\s+(\d{4})$", r"\1", regex=True)
postcode_quality = {
    "invalid_format_before": int((~postcode_text.str.fullmatch(r"(?:\d{4})?")).sum()),
    "invalid_format_after": int((~cleaned["postcode_normalized"].str.fullmatch(r"(?:\d{4})?")).sum()),
    "rows_normalized": int(cleaned["postcode_normalized"].ne(raw["PCODE"]).sum()),
    "missing_retained": int(cleaned["postcode_normalized"].eq("").sum()),
}

format_validation = {
    "blank_address": int(raw["Station_address"].str.strip().eq("").sum()),
    "invalid_charger_type": int((~raw["Charger_Type"].isin(["AC", "DC", "Upcoming"])).sum()),
    "invalid_nonblank_postcode": postcode_quality["invalid_format_after"],
    "invalid_nonblank_objectid": int((~raw["OBJECTID"].str.fullmatch(r"\d*")).sum()),
}
assert not any(format_validation.values()), format_validation
display(pd.DataFrame.from_dict(numeric_validation, orient="index"))


Operators: 50 raw -> 43 canonical


,missing,non_numeric,out_of_range_or_non_integer,invalid_total,validated_dtype,output_column
Latitude,0,0,0,0,Float64,latitude_numeric
Longitude,0,0,0,0,Float64,longitude_numeric
Number_of_plugs,0,0,0,0,Int64,number_of_plugs_numeric


## 3. Charger-rating normalization

Preserve `Charger_rating_raw`; expose normalized scalar power only where justified.
Multi-plug configurations are written to a separate child table, preserving the
one-to-many relationship between a charger and its rated plug groups.


`AC` repeats the charger type and supplies no power information: it remains genuinely
unknown. A mixed configuration has known component powers but no justified single
scalar rating. Its blank scalar is structural missingness, not lost information.
Reported plug totals that differ from component totals remain flagged; the component
description may cover only part of a site. No scalar average or total kW is invented.


In [4]:
PROPER_KW = r"^(\d+) kW$"
UNITLESS_KW = r"^(\d+)$"
MULTI_PLUG = r"^(\d+)x(\d+)kW\s*&\s*(\d+)x(\d+)kW$"

cleaned.insert(cleaned.columns.get_loc("Charger_rating") + 1, "Charger_rating_raw", cleaned["Charger_rating"])
raw_rating = cleaned["Charger_rating_raw"]
proper = raw_rating.str.fullmatch(PROPER_KW)
unitless = raw_rating.str.fullmatch(UNITLESS_KW)
ac_placeholder = raw_rating.eq("AC")
multi_parts = raw_rating.str.extract(MULTI_PLUG)
multi = multi_parts[0].notna()
assert (proper.astype(int) + unitless.astype(int) + ac_placeholder.astype(int) + multi.astype(int)).eq(1).all()

normalized = pd.Series(pd.NA, index=cleaned.index, dtype="string")
rating_kw = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
genuinely_unknown = pd.Series(False, index=cleaned.index, dtype="boolean")
missing_reason = pd.Series("not_missing", index=cleaned.index, dtype="string")
representation = pd.Series(pd.NA, index=cleaned.index, dtype="string")
implied_plugs = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
reported_minus_implied = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
scope_ambiguous = pd.Series(pd.NA, index=cleaned.index, dtype="boolean")

for mask, pattern in [(proper, PROPER_KW), (unitless, UNITLESS_KW)]:
    values = raw_rating.loc[mask].str.extract(pattern)[0].astype("Int64")
    normalized.loc[mask] = values.astype("string") + " kW"
    rating_kw.loc[mask] = values
    representation.loc[mask] = "single_power_kw"

representation.loc[ac_placeholder] = "unknown"
genuinely_unknown.loc[ac_placeholder] = True
missing_reason.loc[ac_placeholder] = "ac_restates_charger_type"

implied_plugs.loc[multi] = multi_parts.loc[multi, 0].astype("Int64") + multi_parts.loc[multi, 2].astype("Int64")
reported = pd.to_numeric(cleaned.loc[multi, "Number_of_plugs"], errors="raise").astype("Int64")
reported_minus_implied.loc[multi] = reported - implied_plugs.loc[multi]
scope_ambiguous.loc[multi] = reported_minus_implied.loc[multi].gt(0)
representation.loc[multi] = "multi_plug_configuration"

cleaned["charger_rating_normalized"] = normalized
cleaned["charger_rating_kw"] = rating_kw
cleaned["rating_genuinely_unknown"] = genuinely_unknown
cleaned["rating_missing_reason"] = missing_reason
cleaned["rating_representation"] = representation
cleaned["rating_implied_plug_count"] = implied_plugs
cleaned["reported_minus_implied_plugs"] = reported_minus_implied
cleaned["plug_count_scope_ambiguous"] = scope_ambiguous

components = []
for index in cleaned.index[multi]:
    values = multi_parts.loc[index]
    for component_index, count_col, power_col in [(1, 0, 1), (2, 2, 3)]:
        components.append({
            "charger_record_id": cleaned.at[index, "charger_record_id"],
            "component_index": component_index,
            "plug_count": int(values[count_col]),
            "power_kw": int(values[power_col]),
        })
charger_power_components = pd.DataFrame(components)
assert len(charger_power_components) == 198
assert not charger_power_components.duplicated(["charger_record_id", "component_index"]).any()
display(cleaned["rating_representation"].value_counts().rename("row_count").to_frame())


# Child-table integrity and rating consistency are checked before exporting.
assert charger_power_components["charger_record_id"].isin(cleaned["charger_record_id"]).all()
assert charger_power_components[["plug_count", "power_kw"]].gt(0).all().all()
assert charger_power_components.groupby("charger_record_id").size().eq(2).all()
assert cleaned.loc[proper | unitless, "charger_rating_kw"].gt(0).all()
assert cleaned.loc[ac_placeholder | multi, "charger_rating_kw"].isna().all()
assert cleaned.loc[ac_placeholder, "Charger_Type"].eq("AC").all()
assert reported_minus_implied.loc[multi].ge(0).all(), "Components exceed the reported plug count."


,row_count
rating_representation,
single_power_kw,1337
unknown,522
multi_plug_configuration,99


## 4. Duplicate classification and completed review

All source rows remain in the audit output. Consumers select rows whose
`duplicate_status != "superseded"`. The existing Figtree and Kempsey decisions stay
in place. The six previously flagged rows are resolved as three overlapping site
pairs, with one representative per pair and an explicit supersession link.

For these three pairs, retain the record with named funding/source provenance and
populated LGA/postcode. This is a documented site-level inference, not proof that
every connector has been identified. Original values and power components stay on
their respective rows; conflicting values are neither combined nor overwritten.

| Reviewed pair | Representative retained | Alternate superseded | Attribute discrepancy retained |
|---|---|---|---|
| Albury, 520 David St | Exploren / Destination Charging R1 | OBJECTID 696 | Four reported plugs and unknown power versus two plugs at 7 kW. |
| Hay, 406 Moppett St | Exploren / Destination Charging R1 | OBJECTID 603 | Four reported plugs and unknown power versus two plugs at 22 kW. |
| UOW, Northfields Avenue | Chargefox / Fast Charging R2 | OBJECTID 552 | Six plugs in both rows, but 175 kW versus a four-plug mixed configuration. |

Evidence reviewed on **17 September 2026**:

- **Albury:** identical source coordinates, site, operator and AC type; address
  differences are formatting only. [AlburyCity's charging inventory](https://www.alburycity.nsw.gov.au/environment/sustainability/electric-vehicle-charging-stations)
  corroborates the David Street Exploren site. It is a current inventory, not a
  December 2025 snapshot, so it does not justify replacing historical plug/power values.
- **Hay:** the same source identity fields agree. [Hay Council's 25 February 2025
  business paper, item C8, page 181](https://www.hay.nsw.gov.au/Portals/0/Files/Council%20Meetings%20and%20Reports/Business%20Paper%2025%20February%202025%20Ordinary%20Meeting.pdf)
  describes four council charging stations alongside NRMA. Stations and plugs are
  different units; this corroborates the site, not a verified plug count or power.
- **UOW:** identical address, coordinates, DC type and six-plug count support overlap.
  [UOW's December 2023 P8 announcement](https://www.uow.edu.au/media/2023/super-fast-ev-chargers-coming-to-uow.php)
  describes one funded installation. Its [campus charging inventory](https://www.uow.edu.au/about/locations/wollongong/getting-to-campus/parking/where-to-park/electric-vehicle-parking/)
  lists six P8 DC ports on Chargefox. Neither confirms the source's 175 kW or mixed
  rating. The separate co-located AC record stays active.

Review is complete for these six records; `duplicate_attribute_conflict` explicitly
retains the power/count uncertainty for all six. Their decision, counterpart, reason
and evidence URLs are also exported in the quality summary. Co-location alone never
triggers a merge: Homebush, Narara and Quest Griffith retain their different
operator/type records. This targeted review does not claim the entire dataset is a
verified census of distinct physical sites.


In [5]:
SUPERSEDED_BY = {
    # Figtree: retain the more detailed multi-plug configuration.
    "chr_cc8db9e86339847395b0df4732f7d27b8b3b033b7f4b65a36fb283fb213c3bb3": "chr_af9c025da56f4cc2d561714de7246dc6bb23686e6b512e6ab05d199a2f57f690",
    # Kempsey: retain known 22 kW rather than the AC placeholder.
    "chr_6f03ded1946613628b5ef5c8e518915afa764f01fad981eac5ca5b5fe77542a0": "chr_b070320b6df294a808c6d40cb378730b74652d2beff3cdf31125d399490e4a40",
}

# Reviewed site overlaps: retain the source-attributed row, without copying values.
DUPLICATE_REVIEW = [
    {
        "site": "Albury, 520 David St",
        "retained_id": "chr_754742a5efb69a89e35202973e172f6a764320d26fc5e47445efadeca031cebe",
        "superseded_id": "chr_10e5473a508a28647a75d953e74487c6c4d9b6cdd4af9348e9e1f8c4d4df9457",
        "conflicting_fields": ["Number_of_plugs", "Charger_rating"],
        "reason": "Same named Exploren AC site, address and coordinates; retain Destination Charging R1 provenance and geographic metadata. Plug/power disagreement remains in the audit rows.",
        "evidence_urls": ["https://www.alburycity.nsw.gov.au/environment/sustainability/electric-vehicle-charging-stations"],
    },
    {
        "site": "Hay, 406 Moppett St",
        "retained_id": "chr_ef926dbadf6ffbe694eac1581b8db7736dca9dc48d956df6cec41f91fbb1d45a",
        "superseded_id": "chr_704a97b0bb05d6c2390ba490cff23347c71eb31df86782a464a32069a221752e",
        "conflicting_fields": ["Number_of_plugs", "Charger_rating"],
        "reason": "Same named Exploren AC site, address and coordinates; retain Destination Charging R1 provenance and geographic metadata. Council evidence corroborates the site, not exact plugs or kW.",
        "evidence_urls": ["https://www.hay.nsw.gov.au/Portals/0/Files/Council%20Meetings%20and%20Reports/Business%20Paper%2025%20February%202025%20Ordinary%20Meeting.pdf"],
    },
    {
        "site": "University of Wollongong, Northfields Avenue",
        "retained_id": "chr_0e54c9e24c9f9ffd735a39ccad36aa314503727e8dbd2aa4ee87ed26bdd12a12",
        "superseded_id": "chr_474bbfa745261fafd0e8a26c2dcd3f315b1b846e270c4ed7f8d9939f7f7b9784",
        "conflicting_fields": ["Operator", "Charger_rating", "rating_implied_plug_count"],
        "reason": "Same six-plug DC site and funding project; retain Chargefox / Fast Charging R2 provenance and geographic metadata. Neither historical power description is verified; keep the separate AC record.",
        "evidence_urls": [
            "https://www.uow.edu.au/media/2023/super-fast-ev-chargers-coming-to-uow.php",
            "https://www.uow.edu.au/about/locations/wollongong/getting-to-campus/parking/where-to-park/electric-vehicle-parking/",
        ],
    },
]
reviewed_ids = [d[key] for d in DUPLICATE_REVIEW for key in ("retained_id", "superseded_id")]
assert len(reviewed_ids) == len(set(reviewed_ids)) == 6
for decision in DUPLICATE_REVIEW:
    SUPERSEDED_BY[decision["superseded_id"]] = decision["retained_id"]

record_ids = set(cleaned["charger_record_id"])
assert set(SUPERSEDED_BY).union(SUPERSEDED_BY.values()).issubset(record_ids)
assert set(SUPERSEDED_BY).isdisjoint(SUPERSEDED_BY.values()), "No self-links or supersession chains."

cleaned["duplicate_status"] = "active"
cleaned["superseded_by"] = ""
superseded = cleaned["charger_record_id"].isin(SUPERSEDED_BY)
cleaned.loc[superseded, "duplicate_status"] = "superseded"
cleaned.loc[superseded, "superseded_by"] = cleaned.loc[superseded, "charger_record_id"].map(SUPERSEDED_BY)

cleaned["duplicate_review_group"] = ""
cleaned["duplicate_resolution"] = "not_applicable"
cleaned["duplicate_attribute_conflict"] = False
for decision in DUPLICATE_REVIEW:
    pair_ids = [decision["retained_id"], decision["superseded_id"]]
    pair = cleaned[cleaned["charger_record_id"].isin(pair_ids)]
    assert len(pair) == 2
    assert pair[["Latitude", "Longitude", "Charger_Type"]].nunique().eq(1).all()
    selected = cleaned["charger_record_id"].isin(pair_ids)
    cleaned.loc[selected, "duplicate_review_group"] = decision["site"]
    cleaned.loc[selected, "duplicate_attribute_conflict"] = True
    for key, label in [("retained_id", "reviewed_representative"), ("superseded_id", "reviewed_superseded")]:
        cleaned.loc[cleaned["charger_record_id"].eq(decision[key]), "duplicate_resolution"] = label

assert len(cleaned) == len(raw)
assert cleaned.loc[cleaned["duplicate_status"].ne("superseded"), "superseded_by"].eq("").all()
assert cleaned["duplicate_status"].eq("superseded").sum() == len(SUPERSEDED_BY)
assert not cleaned["duplicate_status"].eq("flagged_manual_review").any()
eligible = cleaned.loc[cleaned["duplicate_status"].ne("superseded")].copy()
dc_eligible = eligible.loc[eligible["Charger_Type"].eq("DC")]
display(cleaned["duplicate_status"].value_counts().rename("row_count").to_frame())
display(cleaned.loc[cleaned["charger_record_id"].isin(reviewed_ids), [
    "duplicate_review_group", "OBJECTID", "Operator", "Number_of_plugs",
    "Charger_rating", "duplicate_status", "duplicate_resolution",
]])
print(f"Eligible records: {len(eligible)}; DC records: {len(dc_eligible)}; 50% target: {(len(dc_eligible) + 1) // 2}")


,row_count
duplicate_status,
active,1953
superseded,5


,duplicate_review_group,OBJECTID,Operator,Number_of_plugs,Charger_rating,duplicate_status,duplicate_resolution
530,"Hay, 406 Moppett St",,Exploren,4,AC,active,reviewed_representative
604,"Albury, 520 David St",,Exploren,4,AC,active,reviewed_representative
967,"University of Wollongong, Northfields Avenue",,Chargefox,6,175 kW,active,reviewed_representative
1061,"University of Wollongong, Northfields Avenue",552,Unknown/missing,6,2x350kW & 2x175kW,superseded,reviewed_superseded
1073,"Hay, 406 Moppett St",603,Exploren,2,22,superseded,reviewed_superseded
1087,"Albury, 520 David St",696,Exploren,2,7,superseded,reviewed_superseded


Eligible records: 1953; DC records: 431; 50% target: 216


## 5. Write auditable integration outputs

`ev_chargers_cleaned.csv` retains all source records, numeric companions and review
flags. `charger_power_components.csv` retains all mixed-power components, including
those belonging to superseded audit rows. Filter components through the selected
parent records before using them for analysis.

`cleaning_quality_summary.json` records counts and rates, validation results, review
decisions and the DC denominator. Before-missing counts describe source blanks;
after-missing counts additionally recognize unknown operator/power placeholders.
The latter can increase because an uninformative string is not usable information.
Mixed-power rows count as known through their components, despite a missing scalar.
The same metrics are also reported for the non-superseded population, with its own
denominator. These are cleaning metrics; augmentation coverage is measured later.


In [6]:
def semantic_missing(frame):
    counts = frame[list(raw.columns)].replace(r"^\s*$", pd.NA, regex=True).isna().sum()
    counts["Operator"] = frame["Operator"].eq("Unknown/missing").sum() + frame["Operator"].str.strip().eq("").sum()
    counts["Charger_rating"] = frame["rating_genuinely_unknown"].sum()
    return counts.astype(int)


def missing_profile(counts, denominator):
    return {
        "denominator": denominator,
        "missing_counts": counts.to_dict(),
        "missing_rates": (counts / denominator).round(6).to_dict(),
    }


missing_after = semantic_missing(cleaned)
missing_eligible = semantic_missing(eligible)
quality_summary = {
    "input_rows": len(raw),
    "output_rows": len(cleaned),
    "unique_charger_record_ids": int(cleaned["charger_record_id"].nunique()),
    "raw_sha256": RAW_SHA256,
    "operator_unique_before": int(raw["Operator"].nunique()),
    "operator_unique_after": int(cleaned["Operator"].nunique()),
    "operator_rows_changed": int(cleaned["Operator"].ne(cleaned["Operator_raw"]).sum()),
    "missing_before": missing_before.astype(int).to_dict(),
    "missing_after": missing_after.to_dict(),
    "completeness": {
        "before_source_blanks": missing_profile(missing_before.astype(int), len(raw)),
        "after_semantic_all_rows": missing_profile(missing_after, len(cleaned)),
        "after_semantic_non_superseded": missing_profile(missing_eligible, len(eligible)),
        "interpretation": "After counts include Unknown/missing operator and AC-as-power placeholders; multi-plug configurations remain known via the child table. No values imputed.",
    },
    "numeric_validation": numeric_validation,
    "format_validation": format_validation,
    "postcode_quality": postcode_quality,
    "rating_representation": cleaned["rating_representation"].value_counts().to_dict(),
    "rating_quality": {
        "unitless_rows_normalized": int(unitless.sum()),
        "unknown_power_before": int(ac_placeholder.sum()),
        "unknown_power_after": int(cleaned["rating_genuinely_unknown"].sum()),
        "scalar_power_missing": int(cleaned["charger_rating_kw"].isna().sum()),
        "known_power_via_components": int(multi.sum()),
        "plug_count_scope_ambiguous": int(cleaned["plug_count_scope_ambiguous"].fillna(False).sum()),
    },
    "duplicate_status": cleaned["duplicate_status"].value_counts().to_dict(),
    "duplicate_quality": {
        "exact_source_duplicate_rows": int(raw.duplicated().sum()),
        "candidate_groups_before": len(duplicate_groups_before),
        "candidate_groups_non_superseded": int(eligible.groupby(duplicate_key_columns, dropna=False).size().gt(1).sum()),
        "candidate_key": duplicate_key_columns,
        "candidate_key_is_physical_identity": False,
        "previously_flagged_records": len(reviewed_ids),
        "reviewed_records": int(cleaned["duplicate_resolution"].ne("not_applicable").sum()),
        "pending_review_records": int(cleaned["duplicate_status"].eq("flagged_manual_review").sum()),
        "reviewed_rows_with_attribute_conflict": int(cleaned["duplicate_attribute_conflict"].sum()),
        "reviewed_on": "2026-09-17",
        "decisions": DUPLICATE_REVIEW,
    },
    "non_superseded_rows": len(eligible),
    "charger_type_counts_before": raw["Charger_Type"].value_counts().to_dict(),
    "charger_type_counts_non_superseded": eligible["Charger_Type"].value_counts().to_dict(),
    "dc_subset": {
        "definition": "Charger_Type == DC and duplicate_status != superseded",
        "raw_dc_rows": int(raw["Charger_Type"].eq("DC").sum()),
        "eligible_dc_rows": len(dc_eligible),
        "target_50_percent": (len(dc_eligible) + 1) // 2,
        "counting_unit": "retained source record, not independently verified physical site",
        "exported_by": "notebooks/data_augmentation.ipynb",
    },
    "multi_plug_component_rows": len(charger_power_components),
    "non_superseded_component_rows": int(charger_power_components["charger_record_id"].isin(eligible["charger_record_id"]).sum()),
}

# Preserve every original value except the explicitly mapped Operator column.
unchanged_columns = [column for column in raw.columns if column != "Operator"]
pd.testing.assert_frame_equal(cleaned[unchanged_columns], raw[unchanged_columns])
assert cleaned["Operator_raw"].equals(raw["Operator"])
assert cleaned["Charger_rating_raw"].equals(raw["Charger_rating"])
assert hashlib.sha256(RAW_CSV.read_bytes()).hexdigest() == RAW_SHA256

cleaned.to_csv(CLEANED_CSV, index=False, encoding="utf-8", lineterminator="\n")
charger_power_components.to_csv(COMPONENTS_CSV, index=False, encoding="utf-8", lineterminator="\n")
QUALITY_JSON.write_text(json.dumps(quality_summary, indent=2), encoding="utf-8")

roundtrip = pd.read_csv(CLEANED_CSV, dtype=str, keep_default_na=False)
assert len(roundtrip) == len(raw)
assert roundtrip["charger_record_id"].equals(cleaned["charger_record_id"])
pd.testing.assert_frame_equal(roundtrip[unchanged_columns], raw[unchanged_columns])
display(pd.DataFrame({
    "missing_before": missing_before,
    "missing_after_semantic": missing_after,
    "missing_before_percent": (100 * missing_before / len(raw)).round(2),
    "missing_after_percent": (100 * missing_after / len(cleaned)).round(2),
}))
print("Wrote:")
for path in [CLEANED_CSV, COMPONENTS_CSV, QUALITY_JSON]:
    print("-", path)


,missing_before,missing_after_semantic,missing_before_percent,missing_after_percent
OBJECTID,1837,1837,93.82,93.82
Station_name,1438,1438,73.44,73.44
Station_address,0,0,0.00,0.00
Operator,0,1,0.00,0.05
Number_of_plugs,0,0,0.00,0.00
Charger_Type,0,0,0.00,0.00
Charger_rating,0,522,0.00,26.66
Latitude,0,0,0.00,0.00
Longitude,0,0,0.00,0.00
LGANAME,121,121,6.18,6.18


Wrote:
- D:\Data Engineering\comp5339-a1\data\interim\ev_chargers_cleaned.csv
- D:\Data Engineering\comp5339-a1\data\interim\charger_power_components.csv
- D:\Data Engineering\comp5339-a1\data\interim\cleaning_quality_summary.json


## Next hand-off

The working pipeline is **notebook-based**. Run each notebook from a fresh kernel,
top to bottom, in this order:

1. **`data_acquisition.ipynb`** supplies the raw charger CSV and national 2026 SA4
   shapefile. This notebook has consumed the CSV without changing it.
2. **This notebook** writes `data/interim/ev_chargers_cleaned.csv` (all 1,958 audit
   records: 1,953 active, five superseded, zero pending review),
   `charger_power_components.csv` (198 audit components) and
   `cleaning_quality_summary.json` (quality counts, rates and review evidence).
3. **`data_augmentation.ipynb`** reads the cleaned CSV, selects exact `DC` records
   excluding superseded rows, and writes `dc_chargers.csv`,
   `dc_chargers_augmented.csv` and `augmentation_summary.json` under `data/interim/`.
   Following this review there are **431 DC records**, so the ≥50% target remains
   **216**. Its code derives the denominator from the input; earlier narrative
   figures of 432 DC records and six pending reviews describe the old output.
   Both committed caches (`data/external/external_charger_details.csv` and
   `data/external/osm_charging_stations.csv`) support reruns without fresh API calls.
4. **`data_transformation_and_storage.ipynb`** reads the augmented CSV and SA4
   shapefile, applies `sql/schema.sql`, performs the SA4 spatial assignment during
   loading, and writes `data/processed/ev_chargers.duckdb`. It currently stores the
   augmented DC subset, not all cleaned records or the power-component audit table.

After changing cleaning decisions, rerun augmentation and then storage to regenerate
dependent artifacts; use the regenerated summaries rather than old saved outputs
or narrative counts. Keep these record IDs, supersession links and uncertainty flags
when selecting records for analysis. Attribute conflicts remain documented even
though the six duplicate-review decisions are complete.